In [1]:
#cria sessão spark e app
import findspark
findspark.init()

import pyspark 
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("FT03").getOrCreate()

cores = spark._jsc.sc().getExecutorMemoryStatus().keySet().size()
print("Trabalhando com:", cores, "core(s)")
spark

Trabalhando com: 1 core(s)


## Senadores despesas

In [9]:
# timeout define quanto tempo esperar o servidor responder antes de abortar a conexão
#https://python-requests.org/python-requests-timeout/

In [6]:

##importa libs
import requests
import pandas as pd
from pyspark.sql.functions import*
from pyspark.sql.types import*
#from pyspark.sql import SQLContext as spark
from datetime import datetime
data_atual = datetime.now().strftime("%Y-%m-%d")

# solicita dados API 
url = "https://adm.senado.gov.br/adm-dadosabertos/api/v1/senadores/despesas_ceaps/2026"
#response = requests.get(url,timeout=(3,5))# timeout define tempo de espera para resposta do servidor 

try:
    # response.raise_for_status()  # Esta método verifica se a resposta é bem-sucedida. (Bem melhor do que verificar o status_code com if)
    #data_sen_despesas = response.json()
    response = requests.get(url,timeout=(3,5))
    response.raise_for_status()
    data_sen_despesas = response.json()
except requests.exceptions.HTTPError as erro:
    print(f"HTTP com erro: {erro}")  # Print do erro HTTP
    raise
except requests.exceptions.RequestException as erro:
    print(f"Ocorreu um erro: {erro}")  # Print erro
    raise

#Recebe dados da API
data_sen_despesas = response.json()

#Schema de dados para despesas senadores
sen_despesas_schema = StructType([
    StructField("id", StringType(), True),
    StructField("tipoDocumento", StringType(), True),
    StructField("ano", StringType(), True),
    StructField("mes", StringType(), True),
    StructField("codSenador", StringType(), True),
    StructField("nomeSenador", StringType(), True),
    StructField("tipoDespesa", StringType(), True),
    StructField("cpfCnpj", StringType(), True),
    StructField("fornecedor", StringType(), True),
    StructField("documento", StringType(), True),
    StructField("data", StringType(), True),
    StructField("detalhamento", StringType(), True),
    StructField("valorReembolsado", StringType(), True)
])



df_spark_sen_despesas = spark.createDataFrame(data_sen_despesas,schema=sen_despesas_schema) 



# realiza : cast em data e valor
#         : dropa colunas
df_spark_sen_despesas = df_spark_sen_despesas.withColumn('dataDoc',to_date('data'))\
.withColumn('valorReembolso',df_spark_sen_despesas.valorReembolsado.cast('float'))\
.drop('data_nc','data_Doc','valorReembolsado','Valor','data')




### Senadores por UF

In [ ]:
##importa libs
import requests
import pandas as pd
from pyspark.sql.functions import*
from pyspark.sql.types import*
from datetime import datetime
data_atual = datetime.now().strftime("%Y-%m-%d")

url = "https://adm.senado.gov.br/adm-dadosabertos/api/v1/senadores?formato=json"
#response = requests.get(url,timeout=(3,5))# timeout define tempo de espera para resposta do servidor 

try:
    # response.raise_for_status()  # Esta método verifica se a resposta é bem-sucedida. (Bem melhor do que verificar o status_code com if)
    response = requests.get(url,timeout=(3,5))
    response.raise_for_status()
    data_sen_uf = response.json()
except requests.exceptions.HTTPError as erro:
    print(f"HTTP com erro: {erro}")  # Print do erro HTTP
    raise
except requests.exceptions.RequestException as erro:
    print(f"Ocorreu um erro: {erro}")  # Print erro
    raise
##schema para dados de senadores
senadores_uf_schema = StructType(fields=([StructField("codigoParlamentar",StringType(),True),
                                          StructField("nomeParlamentar",StringType(),True),
                                          StructField("titularSuplente",StringType(),True),
                                          StructField("uf",StringType(),True),
                                          StructField("partido",StringType(),True),
                                          StructField("endereco",StringType(),True),
                                          StructField("telefones",StringType(),True),
                                          StructField("fax",StringType(),True),
                                          StructField("dataNascimento",StringType(),True),
                                          StructField("email",StringType(),True),
                                          StructField("chefeGabinete",StringType(),True),
                                          StructField("aniversario",StringType(),True),
                                          StructField("mandato",StringType(),True),                                          
                                          ]))




# cria Dataframe Spark, aplicando o schema(sen_despesas_schema ) 
df_spark_sen_uf =  spark.createDataFrame(data_sen_uf["data"],schema=senadores_uf_schema)
df_spark_sen_uf = df_spark_sen_uf.withColumn('dataNascimento_NC',to_date('dataNascimento'))


### Média de moradores em domicílios particulares permanentes - IBGE

In [ ]:
##importa libs
import requests
#import pandas as pd
#from pyspark.sql.functions import*
#from pyspark.sql.types import*
import pyspark.sql.functions as sq
import pyspark.sql.types as st
from datetime import datetime
#data_atual = datetime.now().strftime("%Y-%m-%d")
 
url = "https://servicodados.ibge.gov.br/api/v3/agregados/579/periodos/-6/variaveis/246?localidades=N3[all]&classificacao=1[6795]"
response = requests.get(url,timeout=(3,5))# timeout define tempo de espera para resposta do servidor 

try:
    # response.raise_for_status()  # Esta método verifica se a resposta é bem-sucedida. (Bem melhor do que verificar o status_code com if)
    #data_sen_despesas = response.json()
    response = requests.get(url,timeout=(3,5))
    response.raise_for_status()
    data_ibge_media_moradores = response.json() 
except requests.exceptions.HTTPError as erro:
    print(f"HTTP com erro: {erro}")  # Print do erro HTTP
    raise
except requests.exceptions.RequestException as erro:
    print(f"Ocorreu um erro: {erro}")  # Print erro
    raise



##schema para dados de senadores
ibge_media_moradores = StructType([
    StructField("id", StringType(), True),
    StructField("variavel", StringType(), True),
    StructField("unidade", StringType(), True),
    StructField("resultados", ArrayType(
        StructType([
            StructField("classificacoes", ArrayType(
                StructType([
                    StructField("id", StringType(), True),
                    StructField("nome", StringType(), True),
                    StructField("categoria", MapType(StringType(), StringType()), True)
                ])
            ), True),
            StructField("series", ArrayType(
                StructType([
                    StructField("localidade", StructType([
                        StructField("id", StringType(), True),
                        StructField("nivel", StructType([
                            StructField("id", StringType(), True),
                            StructField("nome", StringType(), True)
                        ]), True),
                        StructField("nome", StringType(), True)
                    ]), True),
                    StructField("serie", MapType(StringType(), StringType()), True)
                ])
            ), True)
        ])
    ), True)
])





# cria Dataframe Spark, aplicando o schema(sen_despesas_schema ) 
df_spark_media_moradores_IBGE =  spark.createDataFrame(data_ibge_media_moradores,schema=ibge_media_moradores)
# parte 1
df_spark_media_moradores_IBGE = df_spark_media_moradores_IBGE.withColumn('resultados',explode('resultados'))

#parte 2
df_spark_media_moradores_IBGE= (
                        df_spark_media_moradores_IBGE
                        .withColumn("classificacao",explode("resultados.classificacoes"))  
                        .withColumn("serie_item",explode("resultados.series"))  
                     



)

#parte 3
df_spark_media_moradores_IBGE = (
           
        df_spark_media_moradores_IBGE
        .select(
             col("id").alias('variavel_id'),
             col('variavel').alias('variavel_nome'),
             col('unidade'),
             col('classificacao.id').alias('classificacao_id'),
             col('classificacao.nome').alias('classificacao_nome') ,           
             explode('classificacao.categoria').alias('categoria_id','categoria_nome'),
             col('serie_item.localidade.id').alias('localidade_id'),
             col("serie_item.localidade.nome").alias("localidade_nome"),
             col("serie_item.localidade.nivel.id").alias("nivel_id"),
             col("serie_item.localidade.nivel.nome").alias("nivel_nome"),
             col("serie_item.serie").alias("serie_map")

        )

        .select("*",explode("serie_map").alias("ano", "valor_str"))
        .drop("serie_map")
        .withColumn('data_agora',current_date())
)

    

In [9]:
from src.ibge_moradores import ibge_moradores
from src.sen_despesas import sen_despesas
from src.sen_por_uf import sen_por_uf

In [10]:
ibge_moradores().show(3)

+-----------+--------------------+-------+----------------+--------------------+------------+--------------+-------------+---------------+--------+--------------------+----+---------+----------+
|variavel_id|       variavel_nome|unidade|classificacao_id|  classificacao_nome|categoria_id|categoria_nome|localidade_id|localidade_nome|nivel_id|          nivel_nome| ano|valor_str|data_agora|
+-----------+--------------------+-------+----------------+--------------------+------------+--------------+-------------+---------------+--------+--------------------+----+---------+----------+
|        246|Média de moradore...|Pessoas|               1|Situação do domic...|        6795|         Total|           11|       Rondônia|      N3|Unidade da Federação|2007|     3.53|2026-11-22|
|        246|Média de moradore...|Pessoas|               1|Situação do domic...|        6795|         Total|           12|           Acre|      N3|Unidade da Federação|2007|     3.95|2026-11-22|
|        246|Média de mor

In [11]:
sen_despesas().show(4)

+-------+--------------------+----+---+----------+------------+--------------------+------------------+--------------------+----------+------------+----------+--------------+
|     id|       tipoDocumento| ano|mes|codSenador| nomeSenador|         tipoDespesa|           cpfCnpj|          fornecedor| documento|detalhamento|   dataDoc|valorReembolso|
+-------+--------------------+----+---+----------+------------+--------------------+------------------+--------------------+----------+------------+----------+--------------+
|2284372|Nota Fiscal Eletr...|2026|  1|      4642|EFRAIM FILHO|Divulgação da ati...|18.302.933/0001-07|THIAGO VASCONCELO...|   1001193|        null|2026-01-21|        1250.0|
|2284374|Nota Fiscal Eletr...|2026|  1|      4642|EFRAIM FILHO|Divulgação da ati...|15.702.663/0001-25|VALDEREDO BORBA D...|        23|        null|2026-01-27|        1000.0|
|2284375|              Fatura|2026|  1|      4642|EFRAIM FILHO|Locomoção, hosped...|07.319.323/0001-91|   PANTANAL VEICULOS| 

In [12]:
sen_por_uf().show(5)

+-----------------+--------------------+---------------+---+------------+--------------------+--------------------+---+--------------+--------------------+--------------------+-----------+-----------+-----------------+
|codigoParlamentar|     nomeParlamentar|titularSuplente| uf|     partido|            endereco|           telefones|fax|dataNascimento|               email|       chefeGabinete|aniversario|    mandato|dataNascimento_NC|
+-----------------+--------------------+---------------+---+------------+--------------------+--------------------+---+--------------+--------------------+--------------------+-----------+-----------+-----------------+
|             5672|           ALAN RICK|        Titular| AC|REPUBLICANOS|SENADO FEDERAL AN...|       (61)3303-6333|  -|    23/10/1976|sen.alanrick@sena...|Maria Janete Sous...|     23/out|2023 / 2031|             null|
|             5982|   ALESSANDRO VIEIRA|        Titular| SE|         MDB|SENADO FEDERAL AN...|(61)3303-9011; (6...|  -|    0

In [ ]:

def gera_parquet():

    sen_por_uf().coalesce(1).write.mode('overwrite').partitionBy('uf').parquet('output/senadores_uf')
    sen_despesas().coalesce(1).write.mode('overwrite').partitionBy('dataDoc').parquet('output/senadores_depesas')
    ibge_moradores().coalesce(1).write.mode('overwrite').partitionBy('localidade_nome').parquet('output/ibge_media_moradores')


In [ ]:
from src.ibge_moradores import ibge_moradores
from src.sen_despesas import sen_despesas
from src.sen_por_uf import sen_por_uf

In [18]:
from src.gera_parquet import gera_parquet

In [19]:
gera_parquet()

In [ ]:
sen_por_uf().summary().toPandas().to_csv('summary_data_quality/senadores_uf_summary.csv')
sen_despesas().summary().toPandas().to_csv('summary_data_quality/sen_despesas_summary.csv')
ibge_moradores().summary().toPandas().to_csv('summary_data_quality/ibge_media_moradores.csv')

In [28]:
from src.gera_summary import gera_summary

In [29]:
gera_summary()